In [2]:
import sys
sys.path.append("..")  # let the notebook import from the project root
from spark_jobs.common.spark_session import get_spark

spark = get_spark("first-iceberg-table")
spark

26/11/10 22:36:15 WARN Utils: Your hostname, Nihal resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/10 22:36:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/m/Ecommerce/ecommerce-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/nafis_sadik_nihal/.ivy2/cache
The jars for the packages stored in: /home/nafis_sadik_nihal/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.postgresql#postgresql added as a dependency
com.oracle.database.jdbc#ojdbc11 added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-28881799-08ef-426b-af1b-3127f9365677;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.1 in central
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found com.oracle.database.jdbc#ojdbc11;23.5.0.24.07 in central
	found org.mongodb.spark#mongo-spark-connector_2.12;10.4.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	

In [2]:
df = spark.createDataFrame(
    [(1, "Nafis", "Dhaka", 92), (2, "Rahim", "Chattogram", 78), (3, "Karim", "Dhaka", 85)],
    ["id", "name", "city", "score"],
)
dhaka = df.filter(df.city == "Dhaka").select("name", "score")
dhaka.show()
df.printSchema()
df.groupBy("city").avg("score").show()

26/11/09 20:01:17 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.
26/11/09 20:01:18 WARN OAuth2Manager: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to http://localhost:8181/api/catalog/v1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release. It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537


+-----+-----+
| name|score|
+-----+-----+
|Nafis|   92|
|Karim|   85|
+-----+-----+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- score: long (nullable = true)

+----------+----------+
|      city|avg(score)|
+----------+----------+
|     Dhaka|      88.5|
|Chattogram|      78.0|
+----------+----------+



In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo")
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo.students (
        id INT, name STRING, city STRING, score INT
    ) USING iceberg
""")
spark.sql("SHOW TABLES IN demo").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|     demo| students|      false|
+---------+---------+-----------+



In [4]:
df.writeTo("demo.students").append()
spark.table("demo.students").show()

+---+-----+----------+-----+
| id| name|      city|score|
+---+-----+----------+-----+
|  1|Nafis|     Dhaka|   92|
|  2|Rahim|Chattogram|   78|
|  3|Karim|     Dhaka|   85|
+---+-----+----------+-----+



In [5]:
spark.sql("UPDATE demo.students SET city = 'Gazipur' WHERE id = 2")
spark.table("demo.students").show()

+---+-----+-------+-----+
| id| name|   city|score|
+---+-----+-------+-----+
|  1|Nafis|  Dhaka|   92|
|  3|Karim|  Dhaka|   85|
|  2|Rahim|Gazipur|   78|
+---+-----+-------+-----+



In [6]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM demo.students.snapshots").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|3642997890864259868|2026-11-09 20:01:28.217|append   |
|8069740870254201291|2026-11-09 20:01:29.983|overwrite|
+-------------------+-----------------------+---------+



In [7]:
first_snapshot = spark.sql(
    "SELECT 8285646629616484222 FROM demo.students.snapshots ORDER BY committed_at LIMIT 1"
).first()[0]
spark.sql(f"SELECT * FROM demo.students VERSION AS OF {first_snapshot}").show()

IllegalArgumentException: Cannot find snapshot with ID 8285646629616484222

In [8]:
spark.stop()

In [3]:
python_rows = spark.table("silver.sellers")
scala_rows = spark.table("silver.sellers_scala")
print(python_rows.count(), scala_rows.count())
print("Only in Python:", python_rows.exceptAll(scala_rows).count())
print("Only in Scala:", scala_rows.exceptAll(python_rows).count())

26/11/10 22:36:24 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.
26/11/10 22:36:24 WARN OAuth2Manager: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to http://localhost:8181/api/catalog/v1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release. It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `silver`.`sellers` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [silver, sellers], [], false
